In [1]:
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from tqdm import tqdm

In [80]:
# Generate heatmaps for each decade
for decade in tqdm(np.arange(1900, 2010, 10)):
    # Read the CSV file for the current decade
    df = pd.read_csv(f'src/data/heatmaps_data/heatmap_data_{decade}.csv', index_col=0)

    # Extract x-axis and y-axis labels
    x_labels = df.columns.tolist()
    y_labels = df.index.tolist()

    # Create the heatmap
    fig = go.Figure(data=go.Heatmap(
        z=df.values,  
        x=x_labels,   
        y=y_labels,   
        colorscale='YlOrRd', 
    ))

    # Add a title
    fig.update_layout(
    title= f'Correlation between history and movies summaries clusters for {decade}s',
    title_x=0.5,  
    title_font=dict(size=18),  
    xaxis_title="History",  
    yaxis_title="Movies" 
    )

    fig.show()
    
    # Save the heatmap as jpeg
    fig.write_image(f'src/data/heatmaps/heatmap_{decade}.jpeg')

  0%|          | 0/11 [00:00<?, ?it/s]

  9%|▉         | 1/11 [00:00<00:04,  2.13it/s]

 18%|█▊        | 2/11 [00:00<00:02,  3.46it/s]

 36%|███▋      | 4/11 [00:00<00:01,  6.89it/s]

 55%|█████▍    | 6/11 [00:00<00:00,  8.39it/s]

 73%|███████▎  | 8/11 [00:01<00:00, 10.45it/s]

 91%|█████████ | 10/11 [00:01<00:00, 11.79it/s]

100%|██████████| 11/11 [00:01<00:00,  8.63it/s]


In [31]:
import pandas as pd
import numpy as np

# Create an empty list to store the data
higher_accuracy = []

# Loop through each decade
for decade in np.arange(1900, 2010, 10):
    # Read the CSV file for the current decade
    df = pd.read_csv(f'src/data/heatmaps_data/heatmap_data_{decade}.csv', index_col=0)

    # Convert all columns to numeric, forcing errors to NaN (in case of non-numeric values)
    df = df.apply(pd.to_numeric, errors='coerce')

    # Use stack() to flatten the DataFrame and filter values greater than 0.4
    filtered_df = df.stack()
    filtered_df = filtered_df[filtered_df > 0.35]

    # Loop through the filtered values and add them to the list
    for (row_label, col_label), accuracy_value in filtered_df.items():
        higher_accuracy.append({
            'Decade': decade,
            'Movies cluster': row_label,
            'History cluster': col_label,
            'Accuracy': accuracy_value
        })

# Create a DataFrame from the higher_accuracy list
higher_accuracy_df = pd.DataFrame(higher_accuracy)

# Save the result to a CSV file
higher_accuracy_df.to_csv('src/data/higher_accuracy_data.csv', index=False)

# Print the resulting DataFrame
print(higher_accuracy_df)


    Decade Movies cluster History cluster  Accuracy
0     1900       Transmit           Opera  0.429967
1     1900          Radio          Flight  0.469110
2     1910         Studio           Movie  0.396519
3     1920          Queen         Fashion  0.399864
4     1920          Queen          Tennis  0.410217
5     1920     Experiment          Rights  0.402239
6     1930     Apocalypse           Music  0.361673
7     1930         Murder           Music  0.361060
8     1940         Horror            WWII  0.350376
9     1940          Train            WWII  0.418378
10    1950          Genre          Actors  0.406278
11    1950          Names   Entertainment  0.353707
12    1960         Family        Footwear  0.532055
13    1960         Family  Counterculture  0.526284
14    1960        Warfare            Moon  0.448306
15    1970        Warrior        Computer  0.368405
16    1970        Warrior           Music  0.412663
17    1980       Survival        Disaster  0.408370
18    1980  

In [42]:
# find the max similarities for each decade
max_similarities = []

# Loop through each decade
for decade in np.arange(1900, 2010, 10):
    # Read the CSV file for the current decade
    df = pd.read_csv(f'src/data/max_similarity_plots_data/bar_plot_data_{decade}.csv', index_col=0)

    # Convert all columns to numeric, forcing errors to NaN (in case of non-numeric values)
    df = df.apply(pd.to_numeric, errors='coerce')

    # Find the maximum value in the DataFrame
    max_value = df.max().max()

    # Find the row and column labels for the maximum value
    row_label, col_label = np.unravel_index(df.values.argmax(), df.shape)

    # Add the result to the list
    max_similarities.append({
        'Decade': decade,
        'History cluster': df.index[row_label],
        'Similarity': max_value
    })

# Create a DataFrame from the max_similarities list
max_similarities_df = pd.DataFrame(max_similarities)

# Save the result to a CSV file
max_similarities_df.to_csv('src/data/max_similarities_data.csv', index=False)


In [79]:
# bar plot
import pandas as pd
import plotly.express as px

# Read the data from CSV
df = pd.read_csv('src/data/max_similarities_data.csv')

df['Cluster'] = df['History cluster'].astype(str)  # Example: Using 'Decade' as cluster name (adjust as needed)

# Create the bar plot
fig = px.bar(df, x='Decade', y='Similarity',
             labels={'Similarity': 'Max Corr in Movies Summaries'}, height=400)

# Add cluster names (Decade or other) on top of each bar
fig.update_traces(text=df['Cluster'], textposition='outside', texttemplate='%{text}', textfont_size=10, textfont_weight='bold')

# Update the colors of the bars
colors = ['#ba3c3c', '#dc6c3a']  # Muted Red and Muted Blue colors
fig.update_traces(marker_color=[colors[i % len(colors)] for i in range(len(df))])

# Hide the legend
fig.update_layout(showlegend=False)

# Show the plot
fig.show()